# Laya Wikiracing: base versus fine-tuned

A fixed Wikispeedia graph, target-disjoint splits, and identical 60-race evaluation. The notebook runs base Laya before training, then full supervised fine-tuning on one T4. It records decision accuracy, race success, and GPU model latency separately. Source: https://snap.stanford.edu/data/wikispeedia.html


In [ ]:
!pip -q install laya==0.3.5 transformers==4.57.6 safetensors==0.6.2 numpy==2.0.2


In [ ]:
import os, base64, io, tarfile, shutil
from pathlib import Path
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
payload = 'H4sIAAKxt2oC/+08a3PbOJLzWb+Cy6u7JROZlmTLSbSrvfPOeCapzSYuJ9m5OpWKBZOQxBNFcghStuPzf7/uBkCClOQkMzu5qxmikljCo7vR6DfgeMfe8X9cstuXnIU8/+ZXaQPZDv0cDE5O68/YPxyMhqNvrNtvvkIrRcFyQP/N77ONnlubItrw6fDZi8HpaPBsfOKNR4Pn8Ln3Tdd+8+0mWkc5C/ix70dJVPi+l939Gvp/dip1/NnZWOr6yND5s2FL/0+HMGwNvqb+r1gerbPD82DaYvHbO3/btn8EERAZ52HELBQFK2QFs1gSWps05LFVFlEcFREXHkzujMJvq3md/+/8f+3/h+OzkTc+OR2fPT/pVP335P+XOctWv4Lz/7T/Pz3VfZX+n4wHo87/fyX//zploVWsuMXyYBVteWiZEQHJBUUDQZqIIi+Dwor5ksUqOghWaRSo2CDaZGleWP8t0kR/XjGxiqNr/XXDipX+nPPeIk83ADeOeVBEAN5SQyH/qVSjGawAAHrkEgH0frg6v3zpX56/f2lNqcvx/UUUc993vZyLNN5yx/UylvOkELPh3Dq2bIxqjlvijpTavV4v5AsrBjb41O0gzmmNw530LGg5L8o8od15OFk4hBknI1ZYXfDbwnFdBbDg+UY4RVTEvAlB8MLJubeIkpDFsZPbM3b0cXD0Yv7U7ls0H+Df8BxAudaRdW/D4cCInS7wXzgK/BEl9MV+UNjECtgTR6JwaA99Kyhz3D5AZPmSw8842kTFdDhSxMCBXbFkbaVJfKdONC2LZRolS5iarEXfuomKFfTRyZdxgSMhYGAJBIlFqgDTySNAolz05WI4F6JjZstue97XHTRuz2kN27IoZtcxh/nUP1Nky+FooZBYUVLPlfRjExwlByR2as3kxLn11JptrUWaW9vGIoS1tf4wVQDnswnxA/g7nFfw9AEBK3noaOh9a83vpjHbXIfM2k60THtixUbjM2dh3yuiHyb3Ejp82D7YHk8C0BE4Ri+MllyQbBCnaJJPAgKkG4Ii9DbkRJAQvsw5r7mpe4B/NIOOPkhz7mzdmi9FG+pWAcSWbnkesww5zhOnsP6tQY8LurJht86wr4b/pzlcw/kX62V5LcBqcKsUfFHGxHWYSYKCcpMurITleXoD0pJFAcjGNfSGqZWk+GMTJazgXpv7zqn1pCLyqTXwhgPoQMsBarEcZo7mAe6qbx1tXckKQxjUAVaHL0+Q+NQHNABb8On7vOSuEoN57ysdv1RWsG4CDV6lq6DUERgoLpqWAlT/LuP2xLKlmZWKL60w2ksc+XEVBStSHtg6y4soQGEXZNGvAY2VgFWyRMEzOIUblktbr9RKzf93BBzkERxxxADo/YK01L+PHuBbS5GB7XTSUV+qGE/KDc+BeMfYxkNllwocOWSTtpEApOEUWNPauJrqKwr3kKEtBZCuJPTwXKVVMDXngQEWOTg7sD/aXFFmMeiWJNOdHZ1M5nO9NSVIymnQv2oXEZz8BrUA7NJMQvMRWguTlDocLftwTNFydZ3mosnTptE0dLwisVpYj5k0wHY8lmU8CZ3SNXmsJ6jdaMsu/CJ19Jg+KYW3mgMbU7I+sQYPNAYyXaKhIt/tNO3YzQqcs5xRE7mFydTlZSkweQH60dhciZszdtHa3gImoB2BSRVZzSkNimflHImrvm7RUwx35kuC9vKrWvtbSoy6/L/L/438/2T0Yuydvjgbn5086/L/31P+X+QsSv4v8v+TwdnJaCf/Hw67/P8r5f+XLMohdLxmglOeD5E8hPJRwo+KMuEW37K4ZBhuQqpIsaNZHgh5ALERDjCxPlQEMLP+VFT5PyBLN/obGqFHMv5UQDi9jXLIvcsMQ0znw7sL//33U3sAsSt+/v71+X/Kb+/f/u3izav/urh6B+n71fnr1xevX737+9ResFhwG2JwBRlirOzOYsJKsoqKNA9WkoxVuVxC4LEA5fBXZUWOSFgGuXbhh+lNgmUAOVuwBS94AqmD8AiInk9lBaxN9GHOltNHuQQ0LhEQ6ECgVxU+zssifZ+ueRJ95LmcFrM75rElBK16EsC4hSBNzfIhO19ES2MyREwbOBE1+7qM4tCnYk1ffREYokEk05OLtA3wZK3HJFzF7VVxoV+lLn0Z2fd6V2/fvv+8Kkzvu/P35zCVVrQrMnbv7YcKjnHaEEQ6Ngxdfnjvf/fqChOV4zVbLmN+fJPmazihY9z0UQUHkqyri3+8evfq7RsAZw+DMQ+D4TP2jAXh82eDYbg4WwQnp8NgEILPWwyDs1Hw4vnIroJ62DakrMIRWRwVzbRkZhR/ICLnLgWq+AnDUIf2d2xBRkhrH6i+FNuN6pBHQ7hEOO5cIZWZIqJ1qnOFRDW9gZRpsVQ0/ATbga6ZrY9A1VBEsOIbhhG5XUA689NMJo1zmSvKnkbSOFfJnhyq0r65jOOjEPL0LIWEhypy05bMtAic2SQGCFPS0beGPhhx/EvEz+wVbn7Dbv2YA8muLuxgdQFwudZf5LRqhoUshcGKBhfLNthD+Ooks5EOgfsC+/UPsFX8Is/THPJySPCTgGFFACBuWEy6FlYSDGkuAvxjFP5x/mC7zfQzCoE7hA8+EStDZCUxB0jN16C08L2iEXpjds1jvUp+mfcbCYZOavUk/bVOYfWI+lalmtesCFa4fzwbFvbB7m6jQNcVE6vij1AJVxQWK+jFSg6NzGgDcymwOUprPVmVcJvT9R73LwFgWGJCQ+ehv3CcpC+RuppAFMKpnBGnyVKuYwUYSfIlejUIUir8OFpzkgaapXA35xAKRap7CH6aIXCQNrH+7NXXaRq7VSYekVg3c3Dady1qQOcMpk0qkZSsnVcYpSMwx6q1FQMOQahTUsWF5sT6WPajq8frel/Nk8dgYTmsYetuwZE5Ss6INbdk5EgFqm30NZl9E0+/wfpkz2lVpi+D2COShslRTkqKuWFnGsJOkzyMShyts1icxlLLXMk+iL5EFSULnqPVIv/ntKonGPlR/QAc8ZI7g36tRH3rudssJgSrMllLCyxmtHIi1z+1ns8bMw38DNx5wAR4sKAMmd3kwwL8SDE8c3eLFnG6jApggY8qidt1nkgDQEQYrAHnGspIwI9qo+A2AJL2oif5iFv9GGUaisQiyQC/FGRlVYUzm2SvLonssYyftnt7mi2xw7KPs70SCcJHtxnuQ8MyS2qU7CwirEpvMtTSMueOIUtqI1QAnVqO3CS4wYUNZzv0BrXCkwQkmQcuWWQQQDgDb9y3xh6Iw4uhwY84FWiRBt6g12ItAjBQN1n4EdYAdCZYnrM7uU2197mWBxgnAs9Osfpe9JqyAGifTi25A6QzXbIw5LcZRBVhCfR+xEuijzOT/Yb2o6dFEH8mXswGrTKaZhDO6SskhdtgOa0ban3VAb+/4UUeBcJkOkhmfRqKdUEKPi5AHJJv13nEc4OPFL2GXBUVZ3VJ9DHG/gym1oRVUDIJBVjpfAQW7udtzUnpPWBNlBQOIM+X6C7NCWqrcFp67rShFcbMatNasZSIZp4E2j8AwkAHAR35eKBb2lkKmlz3T3KkKRANzyLPoBIqx8lg+7TItZ48sUYQpJYbvTNOe8YO/Duks6np78vCtnmKIHHM+vPU6EPhw0NAAo2ZgKxlJti1QBHfcJY4M2P9z8QJscvRHkNUYVimabgDmzo/E7zbgo6wIAK+1pa2YVbA7sCfoTubHA1BSvePDSfzNlBEntw5ewj4JF/cVlSbgL0lKTEMJcaeaTb0WQDhKAvuYIoW5OPdyc14lgQJFkiBOt4HG8THHw786whxS3mDLsJaqyQMGd+q+xVMvbWZUbGBERVA1qDulLXn69OSphnqW6s08+Xd9/N/eghB7yUpgti5fWjddWHoAJMoX8op5JdfdaTfWKvum5Cgxn14pfoR3okYBrMZ1vCsjmqq3e9x7SBZCjyaGUUnQMBrlea9U0XGZK9jv4YEd70zUmdqaEA+8T7B3Q08wAFMdSpmU24PUiPTzcmj94ruwfCjzp0nj13CYoJcp5kTY8iIdwaHsdR5XkVZneCpO+KdxbIIgIf+iXLAzkopqRhieuIuCVZ5ikuc3Ykke4QCy20e6MjCD9IyKfCxyc7sXx7IPhrMztSG518Y0H7RjpWmVF52z77B9ym+gEOyhoPBYBcMBhxyI7OBSqEMcZk3g2iPCovOobCjZSAAcg1qhrHFxzq2mO+sUqZBb0gBcT9Ltz9XedshvzRZk7YBa4s09GDtFr4rIh/NAmxRBmA0Ra0kNaEACeyWUN5K377jW51PgEQjA1rtq9WK4EavTFkgfQ3u/A1OUiKi8gwUEDKshVH3kO6hyhbymQlhrn0GVctm+c46OApYobc7n+/3yXIymp4bWbbDTvyouvxcGr6qW/lbva7pmTHAof36EKorgFUKIaMfoEkypCYYJ0JUgxTTdngsuPUmTfg+6PwW9/PZSI6ICc2j+HLMNd/9bDww8YGe4cuOKOak5GBxxgOMoxtLXowfX/Ji7LY5WZBvR2EiRuvgZMOiREcD8ibDE5yHzukIAnCAvdMnTdaGJSWLfd3dq42Z4IUPhsMvVlgtFs5pVSlFd2wYvEj41cMmZ6cCegVWDfYia6D236hSbn374btz64fLD7CHn0q87VHlzrcf3nubdRjljqrS06OovsVvwVH76Vq+kVIPzbYy+5GUyK/SHUwGChyZdx/Aob9v35PA3DQBuqMkSbd0nySocm/TiyzKKae6cF8fAYvj9MbPsN6UJ2I6s2XsZly3YFCQxz7dkKi7EPmuE4NP8jD5cau7cjfHT3ShbO+dilNtSBVKF/iip/0AtJp0iJDW61AFSd9jqbr3dDwcQaxq1MqnwxfQwzaZL52tvciGZ3YjuJ3Ohh6WM/DvvDHiX9/5siAnpvfatOntwSYaF00eXgP5ELjTJTBI5s6+qqW22gDdYtAzoIRt+MSafSJmMVP6+ooF1+74OJyJA1RttIkkep7G4ug6J8GhMwQrYrsyispydJv25dXF5fnVxXcwfL/md8p54I2ARA99fYu+I2i5Aw88y0Y47oOp9XQXQbtA6aqK4nWhVu9GGIAIMEBq7FTWEhdxKVaGLsl7tqn55FjlIUyWQQwR05dKNqYrPk04KFXyRfbUvPJzKEtSeoCHORVFvnu+agKcrmtmSLLXU8Kcc50KBekmi+jR7vd4p2osoS1RbO5jFuhU15/OXry72uyizRaY8xUGx+S8uh4tS0lMcLxI80E0gJQ9lWR5NLOG7Mz3lZab8FCPAGC7vGjia61A3OJxGkhgPwO5LrAJekvXKrY1EfabJLcAaWH6+Wn0HugO3tKCNOohLYs3eIsohZGEN4QQWEDoWO0GqwQ7W+zLvK4xSB0PfXr7nBTTkZJIpeN/PX938frVmwvQy5+H534N4ak0B+r9ahOzNgjoeNd48agDgIeHPZrc1BPgYhiR9V/xYJ2lQDFElj5PyFUfGF3fQMgLRtouBZDAYQKEDhhUk2o91IVqSuolQhZABAXaj5frYCnBNwozEsDyJfl5gdYl9B0Cpcp5klaMVDNpavsmZDS8oW+CpQjEo6BfYEboaGPh2S69F2lhk+YcqftCHBDlfDke9HEb5dNkZEI93nnINj86s3ub0ODBK2gYEGLJasSPxiBlxgQkWY+OcRSU9Qbf1hYoSOxu6g2GiodZGqxIPyCP9QWgBz8dBOUGiDjpW6fwRzncAmO9gmeoh3KVfsoe8Ch26k/oYCpLQS5vjgF+jQG/EQ63uvQPy3hn33HuV0Pea3qz/vrKqbjUt9Q7diRqYkHi4zhUL3pqDY2X/6hqJvFPLG9wZkbGOG8AYUdj1hFBNcC0QLw4dbV/EWBCDdohyPF+gDN9R92qyCBnriASTfO7utT16RoGShwxu66CSd4bKqIcSk5RfM8oOVGtqnkQ4P8wnTcCbRnar8rFIubGrXtDIKkoT5LqYGhfpH4C6YxhO/TtDm/V8XQNr2+li4WQv/ti3EXvuas0pOSxW0sJbqKgPjWW/b++wDQu3yTeJPEWZUJelsVekMOYj2YTT6l5qdlv3pDP8upCpM48iaS5xj3dS0IjCYvEAv+rBE53ZvsuTPfkYW8wgsJFtBO7CV0qg0c/CGil6t41C9A/hK3KlhQbXaCRmyVqdsg2tPtftY2aWgOs8u6RA+svxkOS3Z0pQsuEPvi1WTlUoIOTwv89QHhBHGXSFyVpvvFlGNLwXsat7D7ewC4MdH+qSJE5FHVoq0dzoefLNHEfv4bjAXFrlxNVxvH2h6uLd+9AMaTBeYr3OBqEoaPW8XHDW+RgsMKqSiKPc3YE+Cb4K02neyINwxbWhTlCir6tRi5rMyRl7WKMROPqG3Vir36S6FBcBGl9wUBNXW/F4kVVz4T4v4iWZVrq9EaHTip8r8N9t86nMIjX0eK+ML/X4uXF5dtvXwIj1SZndEnW5gJZZIyaBAei5COkxyq6qnQRW4YHrCSlr6So16ooQ7Bb3EGQD/OU3uFD2PCflWZIYAdyjAqTOfeXZxcSzqOphYGqb9Borv/lGUUbLr3821+2sPHO+n6hfvNtcn9dBmte4O+j1UA+9cqkVVWQIGRdYYRFhJOjMf44Oxriu2F7OHyqSwpE2IJyH8RmN15IyfeKlMZiLEuDduMVKn1Qb6TtRuJ0oEK0P38iTrYSodp3kvIaVRuFwajVHM7lD6yt6gCqRk0vgfESDnW8pN83xNw/Bj2m30LG3MnX5TsYrQp4lo0lTug5HcHnZVbiudUqhr+tJ0WDOOgMdq7s7LaqI4BWF14TSGuB4bv8tAMHScS876ukiLu7IOlpo2+p46O4DdX7EsRkGMjbaLBUDWva71YN7KEpqXT6jwuonLMjoxUINBRFo2h1qFCgjqnJ6ZneIN02KVYaDKnHH1qYK/Nm2MwvJUHZwhZi6n1oViW+ffv3y9cX76n0uMOdljPv9eD0fBJ738fwwvZ9vEzwfVu9TpA5Sfe7SV3rWte61rWuda1rXeta17rWta51rWtd61rXuta1rnWta13rWte61rWuda1rXeta17rWta51bX/7X29jnfMAeAAA'
root=Path('/kaggle/working/wikirace-project')
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as archive:
    archive.extractall(root, filter='data')
(root/'data/wikirace').mkdir(parents=True,exist_ok=True)
for source in Path('/kaggle/input/laya-wikirace-data').iterdir():
    if source.is_file() and source.suffix in ('.json','.jsonl'):
        shutil.copy2(source,root/'data/wikirace'/source.name)
os.chdir(root)
print('Graph and splits unpacked')


In [ ]:
!PYTHONPATH=. OUTPUT_DIR=/kaggle/working/laya-wikirace python -u -m wikirace.train


In [ ]:
import json
from pathlib import Path
report=json.loads(Path('/kaggle/working/laya-wikirace/report.json').read_text())
print(json.dumps(report, indent=2))
